# Step 4 — Train Regression Model
**Course:** ETL (G01) — Workshop 3  
**Goal:** Train a regression model to predict `happiness_score`, evaluate it and serialize it as `model.pkl`.

### Model selection (based on real evaluation results)

| Model | MAE | RMSE | R² | Decision |
|-------|-----|------|----|----------|
| LinearRegression | 0.4486 | 0.5843 | 0.727 | Decent baseline |
| **RandomForestRegressor** | **0.4069** | **0.5237** | **0.780** | ✅ **Selected** |
| DecisionTreeRegressor | 0.6002 | 0.7891 | 0.501 | Overfits, worst R² |

**RandomForestRegressor** is selected because it achieves the lowest MAE and RMSE and the highest R², while still being a simple model aligned with the workshop's goal of pipeline integration over model complexity.

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os

from sklearn.linear_model    import LinearRegression
from sklearn.ensemble        import RandomForestRegressor
from sklearn.tree            import DecisionTreeRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing   import StandardScaler
from sklearn.metrics         import mean_absolute_error, mean_squared_error, r2_score

sns.set_theme(style='whitegrid', palette='muted')
pd.set_option('display.max_columns', None)

RANDOM_STATE = 42

## 2. Load ML-Ready Dataset

In [ ]:
# ------------------------------------------------------------------
# Load the dataset produced by feature_engineering.ipynb
# This file already has scaled features and the target column
# ------------------------------------------------------------------
df = pd.read_csv('../data/processed/happiness_ml_ready.csv')

print(f'Shape: {df.shape}')
display(df.head())

## 3. Define Features and Target

In [ ]:
# ------------------------------------------------------------------
# Feature order is critical — the Kafka consumer must send features
# in this exact order when calling model.predict()
# ------------------------------------------------------------------
FEATURES = ['gdp', 'family', 'health', 'freedom', 'generosity', 'corruption']
TARGET   = 'happiness_score'

X = df[FEATURES]
y = df[TARGET]

print(f'X shape: {X.shape}')
print(f'y shape: {y.shape}')
print(f'Feature order: {FEATURES}')

## 4. Train / Test Split

In [ ]:
# ------------------------------------------------------------------
# 70% training — 30% testing as suggested by the workshop
# random_state=42 ensures reproducibility
# ------------------------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3,
    random_state=RANDOM_STATE
)

print(f'Training set : {X_train.shape[0]} rows ({X_train.shape[0]/len(X)*100:.1f}%)')
print(f'Test set     : {X_test.shape[0]}  rows ({X_test.shape[0]/len(X)*100:.1f}%)')

## 5. Train and Compare All Three Models

In [ ]:
# ------------------------------------------------------------------
# Train all three candidate models and collect metrics
# ------------------------------------------------------------------
candidate_models = {
    'LinearRegression':      LinearRegression(),
    'RandomForestRegressor': RandomForestRegressor(n_estimators=100, random_state=RANDOM_STATE),
    'DecisionTreeRegressor': DecisionTreeRegressor(random_state=RANDOM_STATE),
}

results = []

for name, model in candidate_models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    mae  = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2   = r2_score(y_test, y_pred)

    results.append({'model': name, 'MAE': mae, 'RMSE': rmse, 'R2': r2})
    print(f'{name:30s} | MAE={mae:.4f} | RMSE={rmse:.4f} | R²={r2:.4f}')

df_results = pd.DataFrame(results)
display(df_results.round(4))

## 6. Visualize Model Comparison

In [ ]:
# ------------------------------------------------------------------
# Bar chart comparing MAE, RMSE, R² across models
# ------------------------------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
metrics = ['MAE', 'RMSE', 'R2']
colors  = ['steelblue', 'tomato', 'seagreen']

for ax, metric, color in zip(axes, metrics, colors):
    ax.bar(df_results['model'], df_results[metric], color=color, edgecolor='white')
    ax.set_title(metric)
    ax.set_ylabel(metric)
    ax.set_xticklabels(df_results['model'], rotation=15, ha='right')
    # Annotate values
    for i, v in enumerate(df_results[metric]):
        ax.text(i, v + 0.01, f'{v:.3f}', ha='center', fontsize=9)

plt.suptitle('Model Comparison — MAE, RMSE, R²', fontsize=14)
plt.tight_layout()
plt.show()

## 7. Train Selected Model — RandomForestRegressor

In [ ]:
# ------------------------------------------------------------------
# Train the selected model on the full training set
# RandomForestRegressor selected: best MAE, RMSE and R² among candidates
# ------------------------------------------------------------------
model = RandomForestRegressor(n_estimators=100, random_state=RANDOM_STATE)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

mae  = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2   = r2_score(y_test, y_pred)

print('=== Final Model Evaluation ===')
print(f'Model : RandomForestRegressor (n_estimators=100)')
print(f'MAE   : {mae:.4f}')
print(f'RMSE  : {rmse:.4f}')
print(f'R²    : {r2:.4f}')

## 8. Predicted vs Actual Plot

In [ ]:
# ------------------------------------------------------------------
# Scatter: predicted vs actual — ideal model follows y=x line
# ------------------------------------------------------------------
plt.figure(figsize=(7, 6))
plt.scatter(y_test, y_pred, alpha=0.4, color='steelblue', s=20)
plt.plot(
    [y_test.min(), y_test.max()],
    [y_test.min(), y_test.max()],
    'r--', linewidth=1.5, label='Perfect prediction'
)
plt.xlabel('Actual Happiness Score')
plt.ylabel('Predicted Happiness Score')
plt.title(f'Predicted vs Actual — RandomForest (R²={r2:.3f})')
plt.legend()
plt.tight_layout()
plt.show()

## 9. Residuals Analysis

In [ ]:
# ------------------------------------------------------------------
# Residuals = actual - predicted
# Good model: residuals centered around 0 with no pattern
# ------------------------------------------------------------------
residuals = y_test.values - y_pred

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Residuals vs predicted
axes[0].scatter(y_pred, residuals, alpha=0.4, color='steelblue', s=20)
axes[0].axhline(0, color='red', linestyle='--', linewidth=1.5)
axes[0].set_xlabel('Predicted Score')
axes[0].set_ylabel('Residual')
axes[0].set_title('Residuals vs Predicted')

# Residual distribution
axes[1].hist(residuals, bins=30, color='steelblue', edgecolor='white')
axes[1].axvline(0, color='red', linestyle='--', linewidth=1.5)
axes[1].set_xlabel('Residual')
axes[1].set_ylabel('Count')
axes[1].set_title('Residual Distribution')

plt.suptitle('Residuals Analysis', fontsize=14)
plt.tight_layout()
plt.show()

## 10. Feature Importances

In [ ]:
# ------------------------------------------------------------------
# Feature importances from the trained RandomForest
# Shows which features contributed most to predictions
# ------------------------------------------------------------------
importances = pd.Series(model.feature_importances_, index=FEATURES).sort_values(ascending=False)

plt.figure(figsize=(8, 4))
importances.plot(kind='bar', color='steelblue', edgecolor='white')
plt.title('Feature Importances — RandomForestRegressor')
plt.ylabel('Importance')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

print('\nFeature importances:')
print(importances.round(4))

## 11. Save Model

In [ ]:
# ------------------------------------------------------------------
# Save the trained model as model.pkl
# The Kafka consumer will load this file to generate predictions
# ------------------------------------------------------------------
os.makedirs('../models', exist_ok=True)
model_path = '../models/model.pkl'
joblib.dump(model, model_path)

print(f'Model saved to: {model_path}')

# Verify the saved model loads and predicts correctly
loaded_model = joblib.load(model_path)
sample       = X_test.iloc[:3]
sample_pred  = loaded_model.predict(sample)

print(f'\nVerification — predictions from loaded model:')
for i, pred in enumerate(sample_pred):
    print(f'  Sample {i+1}: predicted={pred:.4f} | actual={y_test.iloc[i]:.4f}')

## 12. Training Summary

In [ ]:
# ------------------------------------------------------------------
# Final summary — useful for the README
# ------------------------------------------------------------------
print('=== Training Summary ===')
print(f'Dataset size      : {len(df)} records (2015–2019)')
print(f'Training set      : {len(X_train)} records (70%)')
print(f'Test set          : {len(X_test)} records (30%)')
print(f'Features used     : {FEATURES}')
print(f'Target            : {TARGET}')
print(f'Model             : RandomForestRegressor(n_estimators=100)')
print(f'MAE               : {mae:.4f}')
print(f'RMSE              : {rmse:.4f}')
print(f'R²                : {r2:.4f}')
print(f'Model saved to    : ../models/model.pkl')
print(f'Scaler saved to   : ../models/scaler.pkl')